<a href="https://colab.research.google.com/github/Joa1Camargo/End-to-End-Data-Solutions/blob/main/RAG_From_Scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple RAG System from Scratch

---

This is a simple AI that uses RAG to prevent hallucinations. It trades the broad knowledge of a massive model for the pinpoint accuracy of a curated dataset.

Large Language Models (LLMs) are like people who have read the entire internet but sometimes misremember details. A RAG model is like a researcher with a very specific, high-quality library—they won't lie to you, but they can't tell you what’s not in the books.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!pip install transformers
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain
!pip install -U langchain-text-splitters
!pip install -q --upgrade transformers
!pip install -q --upgrade transformers huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 10.4 MB/s eta 0:00:00


Now its time to upload my knowledge to the LLM. I cant feed it all at once to the model, it will be splitted in chunks by the method below:

In [5]:
path = '/content/drive/MyDrive/Colab Notebooks/Datasets/RAG From Scratch/'

In [6]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter

#time to load the knowledge
with open(path+'brazil_geography.txt') as f:
  knowledge_text = f.read()

# 1. Initialize the Text Splitter
#This splitter is smart. It tries to split on paragraphs ("\n\n"),
#then newlines ("\n"), then spaces (" ") to keep semantically related text
#together as much as possible

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350, #max size of a chunk
    chunk_overlap=20, #overlap between chunks
    length_function=len,
)

#2. Create the chuncks
chunks = text_splitter.split_text(knowledge_text)

print(f"We have {len(chunks)} chunks:")
for i, chunk in enumerate(chunks):
  print(f"--- Chunk {i+1}---\n {chunk}\n")

We have 42 chunks:
--- Chunk 1---
 GEOGRAPHY OF BRAZIL: COMPREHENSIVE KNOWLEDGE BASE FOR RAG MODELS

--- Chunk 2---
 1. PHYSICAL DIMENSIONS AND GEOPOLITICAL BORDERS

--- Chunk 3---
 Brazil is the largest country in South America and the fifth largest in the world by land area. It occupies approximately 47% of the South American continent, covering an area of 8,515,767 square

--- Chunk 4---
 of 8,515,767 square kilometers. It shares a land border with ten countries: Argentina, Bolivia, Colombia, Guyana, Paraguay, Peru, Suriname, Uruguay, Venezuela, and the French overseas territory of

--- Chunk 5---
 territory of French Guiana. The only South American countries Brazil does not border are Chile and Ecuador.

--- Chunk 6---
 Coastal Geography: Brazil has an extensive coastline of 7,491 km along the Atlantic Ocean. Its territorial waters include several island groups, most notably Fernando de Noronha, the Abrolhos

--- Chunk 7---
 the Abrolhos Archipelago, and Trindade and Martim Vaz.

-

Now its time to turn the text into numbers (vectors), so the model can read it.

In [7]:
from sentence_transformers import SentenceTransformer

#1. Load the embedding model
#all-MiniLM-L6-v2' is a fanstastic, fast and small model. Runs 100% on local machine.
modelembedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

#2. Embed all our chunks
#This will take a moment as it 'reads' and 'understands' each chunk
chunk_embeddings = modelembedder.encode(chunks)

print(f"Shape of our embeddings: {chunk_embeddings.shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of our embeddings: (42, 384)


Now the vectors need to be stored in a database. I will use FAISS.

FAISS (Facebook AI Similarity Search) is an open-source library specifically designed for the efficient search and clustering of dense vectors.

In [8]:
import faiss
import numpy as np

#get the dimension of the vector
d= chunk_embeddings.shape[1]

#1. Initialize the index
#IndexFlat2 is the simples, most basic index. It calculates
#the exact distance (L2 distance) between our query and all vectors
index = faiss.IndexFlatL2(d)

#2 Add our chunk embeddings to the index
# We must convert to float32 for FAISS
index.add(np.array(chunk_embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors.")

FAISS index created with 42 vectors.


Now its time to build the RAG pipeline.

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load tokenizer & model directly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# --- This is our RAG pipeline function ---
def answer_question(query):
    # 1. RETRIEVE
    query_embedding = modelembedder.encode([query]).astype('float32')
    k = 2
    distances, indices = index.search(query_embedding, k)
    retrieved_chunks = [chunks[i] for i in indices[0]]
    context = "\n\n".join(retrieved_chunks)

    # 2. AUGMENT
    prompt_template = f"""Answer the following question using only the provided context.
If the answer is not in the context, say "I don't have that information."

Context:
{context}

Question:
{query}

Answer:"""

    # 3. GENERATE
    inputs = tokenizer(prompt_template, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    answer_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"--- CONTEXT ---\n{context}\n")
    return answer_text

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [10]:
query_1 = "What is the highest peak in Brazil, and where is it located?"
print(f"Query: {query_1}")
print(f"Answer: {answer_question(query_1)}\n")

Query: What is the highest peak in Brazil, and where is it located?
--- CONTEXT ---
Highest Peak: Pico da Neblina, located in the Guiana Highlands (Amazonas state), reaching 2,994 meters above sea level.

4. TOPOGRAPHY AND RELIEF
Brazil’s landscape is old and stable, lacking the high-altitude mountain ranges found in the Andes. The relief is generally divided into three categories:

Answer: Pico da Neblina



In [11]:
query_2 = "Which Brazilian biome is characterized as a semi-arid shrubland exclusive to the Northeast?"
print(f"Query: {query_2}")
print(f"Answer: {answer_question(query_2)}\n")

Query: Which Brazilian biome is characterized as a semi-arid shrubland exclusive to the Northeast?
--- CONTEXT ---
- CAATINGA: A semi-arid biome exclusive to the Brazilian Northeast. Vegetation consists of drought-resistant shrubs and cacti. It experiences high temperatures and seasonal rainfall.

- CERRADO: A vast tropical savanna region in the central highlands. It is the second-largest biome in Brazil and is vital for water distribution across the country's river basins.

Answer: CAATINGA



In [12]:
query_3 = "What are the two South American countries that do not share a border with Brazil?"
print(f"Query: {query_3}")
print(f"Answer: {answer_question(query_3)}\n")

Query: What are the two South American countries that do not share a border with Brazil?
--- CONTEXT ---
territory of French Guiana. The only South American countries Brazil does not border are Chile and Ecuador.

Brazil is the largest country in South America and the fifth largest in the world by land area. It occupies approximately 47% of the South American continent, covering an area of 8,515,767 square

Answer: Chile and Ecuador



In [16]:
query_4 = "Which river basin is home to the Itaipu Dam?"
print(f"Query: {query_4}")
print(f"Answer: {answer_question(query_4)}\n")

Query: Which river basin is home to the Itaipu Dam?
--- CONTEXT ---
- PARANÁ BASIN: Part of the Platine system, it houses the Itaipu Dam, one of the world's largest hydroelectric power plants.

- SÃO FRANCISCO RIVER: Known as the "River of National Integration," it crosses the Southeast and Northeast regions, providing vital water for irrigation and hydroelectricity in semi-arid zones.

Answer: PARAN BASIN



In [15]:
query_5 = "What is the current population of the city of São Paulo according to the 2022 census?"
print(f"Query: {query_5}")
print(f"Answer: {answer_question(query_5)}\n")

Query: What is the current population of the city of São Paulo according to the 2022 census?
--- CONTEXT ---
- SOUTHEAST REGION (Região Sudeste): Comprising four states (Espírito Santo, Minas Gerais, Rio de Janeiro, and São Paulo). It is the economic engine of Brazil, containing the largest cities and the

cities and the highest population density.

Answer: I don't have that information

